# Análise Exploratória — Índices de Pobreza no Brasil
Fonte: [Kaggle](https://www.kaggle.com/datasets/fidelissauro/indices-pobreza-brasil). Rode as células em ordem; ajuste as variáveis marcadas com `# AJUSTAR` depois de ver as colunas reais.

In [ ]:
import sys
from pathlib import Path
import pandas as pd, numpy as np
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import etl

ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

## 1. Dados brutos

In [ ]:
raw = etl.read_raw()
display(raw.head())
raw.info()

## 2. Limpeza (via `src/etl.py`)
O relatório mostra o que mudou: duplicatas, nulos, conversões de tipo.

In [ ]:
df, report = etl.clean(raw)
etl.save(df, report)
report

## 3. Visão geral

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols and not np.issubdtype(df[c].dtype, np.datetime64)]
print("Numéricas:", num_cols); print("Categóricas:", cat_cols)
display(df[num_cols].describe().T)
display(df.isna().mean().sort_values(ascending=False).rename("proporção_nulos").to_frame())

## 4. Variáveis de análise
Escolha a coluna de grupo (ex.: estado/região), a de tempo (ex.: ano) e a métrica principal.

In [ ]:
GRUPO  = cat_cols[0] if cat_cols else None            # AJUSTAR
TEMPO  = next((c for c in num_cols if "ano" in c), None)  # AJUSTAR
METRICA = next((c for c in num_cols if c != TEMPO), None) # AJUSTAR
GRUPO, TEMPO, METRICA

## 5. Distribuição e outliers (IQR)

In [ ]:
fig1 = px.histogram(df, x=METRICA, nbins=30, marginal="box", title=f"Distribuição de {METRICA}")
fig1.show(); fig1.write_html(ASSETS / "fig1_distribuicao.html", include_plotlyjs="cdn")

q1, q3 = df[METRICA].quantile([.25, .75]); iqr = q3 - q1
outliers = df[(df[METRICA] < q1 - 1.5*iqr) | (df[METRICA] > q3 + 1.5*iqr)]
print(f"{len(outliers)} outliers em {METRICA}"); outliers.head()

## 6. RQ1 — padrão ao longo do tempo (ou ranking, se não houver coluna de tempo)

In [ ]:
if TEMPO and GRUPO:
    g = df.groupby([TEMPO, GRUPO], as_index=False)[METRICA].mean()
    fig2 = px.line(g, x=TEMPO, y=METRICA, color=GRUPO, title=f"{METRICA} por {GRUPO} ao longo de {TEMPO}")
elif GRUPO:
    g = df.groupby(GRUPO, as_index=False)[METRICA].mean().sort_values(METRICA)
    fig2 = px.bar(g, x=METRICA, y=GRUPO, orientation="h", title=f"{METRICA} médio por {GRUPO}")
fig2.show(); fig2.write_html(ASSETS / "fig2_rq1.html", include_plotlyjs="cdn")

## 7. RQ2 — relação entre indicadores (correlação)

In [ ]:
corr = df[num_cols].corr()
fig3 = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlação entre indicadores")
fig3.show(); fig3.write_html(ASSETS / "fig3_correlacao.html", include_plotlyjs="cdn")

## 8. Achados
Escreva aqui 3 a 5 frases com os números que respondem RQ1 e RQ2. Elas viram a seção 3 do artigo.